# Chạy benchmark B0–B4 trên Colab L4 (eval)

Đầu vào trên Drive `MyDrive/bv-rag/`: `processed/{VER}/`, `vectors/{VER}/` (đã đóng băng theo `data/manifests/{VER}.json`).
Đầu ra: `runs/{run_id}/log.jsonl` + `meta.json` ghi thẳng lên Drive. Metric tính trên WSL:
`scripts/sync.sh pull runs` rồi `PYTHONPATH=. python -m eval.metrics runs/<run_id> ...`.
Notebook chỉ điều phối; logic ở `src/pipeline.py`, `eval/run_eval.py`.

1. **Runtime → Change runtime type → GPU L4 → Save → Connect.** Cell 1 dừng nếu GPU không phải L4.
2. Sửa tham số ở cell 2 (`BRANCH`, `CONFIGS`), rồi **Runtime → Run all**.
3. Colab ngắt giữa chừng: chạy lại cell 1–5, rồi ở cell 6/7 dùng `--resume <run_id>` cho run dở (1 config mỗi lần); câu đã có trong log được bỏ qua.
4. Xong: cell 8 in tóm tắt meta (phải `gpu=L4`, `llm_gpu_pct=100`, `ctx_overflow=0`), cell 9 tắt runtime.
Trước khi commit notebook: Edit → Clear all outputs.

## 1. Kiểm tra GPU (bắt buộc L4)

In [ ]:
import subprocess
GPU = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
print(GPU)
assert "L4" in GPU, f"GPU là {GPU}, không phải L4: đổi runtime trước mọi lần chạy chính thức (decision-log 2026-10-25)"

## 2. Tham số, mount Drive

In [ ]:
import os
VER = "v1"
BRANCH = "main"              # nhánh code
CONFIGS = "configs/B4.yaml configs/ablation/B4rr.yaml"   # tuần 6 (dev): B4 min_score null + ablation rerank với rag_v1
TEST_CONFIGS = CONFIGS + " configs/ablation/B4ng.yaml"     # test: thêm B4 không chặn; chỉ chạy sau khi ghi min_score vào B4.yaml
DEV_REPEAT = 3               # dev chạy 3 lần (seed 42, 43, 44) để đo dao động LLM

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/bv-rag"
os.environ.update(DRIVE_DIR=DRIVE_DIR, QDRANT_URL=":memory:", OLLAMA_URL="http://localhost:11434", PYTHONPATH="/content/bv-rag")
!ls {DRIVE_DIR}/processed/{VER} {DRIVE_DIR}/vectors/{VER}

## 3. Clone repo, cài thư viện
Torch bản CUDA có sẵn trên Colab. Nếu pip báo cần khởi động lại: Runtime → Restart session, rồi chạy tiếp từ cell 2.

In [ ]:
!rm -rf /content/bv-rag && git clone -q -b {BRANCH} https://github.com/daisyngocha/bv-rag.git /content/bv-rag
%cd /content/bv-rag
!git branch --show-current && git log --oneline -1
!pip -q install -r requirements/base.txt
assert os.path.exists("eval/run_eval.py"), f"Nhánh {BRANCH} chưa có eval/run_eval.py: kiểm tra BRANCH ở cell 2"

## 4. Kiểm tra kho theo manifest
`processed/{VER}` và `vectors/{VER}` trên Drive phải khớp sha256 trong `data/manifests/{VER}.json`; sai thì dừng.

In [ ]:
MANIFEST = f"data/manifests/{VER}.json"
assert os.path.exists(MANIFEST), f"{VER} chưa đóng băng: không chạy benchmark chính thức trên dữ liệu chưa đóng băng"
r = subprocess.run(["python", "scripts/freeze_data.py", "verify", VER, "--processed", f"{DRIVE_DIR}/processed/{VER}",
                    "--vectors", f"{DRIVE_DIR}/vectors/{VER}"], capture_output=True, text=True)
print(r.stdout, r.stderr)
assert r.returncode == 0, f"Drive processed/vectors {VER} không khớp {MANIFEST}: không chạy tiếp"

## 5. Cài Ollama, bật server, tải model
`pciutils`, `lshw` phải có trước để Ollama nhận GPU. Dòng log cuối phải có `cuda`.

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import time, requests
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2); break
    except requests.RequestException:
        time.sleep(1)
!ollama pull qwen2.5:7b-instruct-q4_K_M
!grep -iE "inference compute|library=cuda" /content/ollama.log | head -3

## 5b. Kiểm độ dài prompt (150 câu, không sinh)
Đếm token mọi prompt của các config trong `CONFIGS` bằng tokenizer Qwen. Phải in `vượt 0 câu` cho mọi config; có câu vượt thì tăng `llm.num_ctx`
ở CẢ 5 config B0–B4 (khối `llm` giống hệt nhau), commit, chạy lại từ cell 3. Cell 6, 7 cũng tự kiểm lại và dừng nếu vượt.

In [ ]:
!python -m eval.run_eval --config {TEST_CONFIGS} --split all --check-only

## 6. Dev: `CONFIGS` × 3 lần
`run_eval` nạp Qwen lên GPU (warmup) TRƯỚC bge-m3, dựng Qdrant `:memory:` một lần cho mọi config, ghi log từng câu lên Drive.
Dòng đầu phải in `LLM trên GPU 100.0%`. 30 câu × số config × 3 lần lượt sinh (tuần 6: B4 + B4rr ≈ 180, cộng rerank ~0,6 s/câu, câu so sánh rerank 2–3 lần).

In [ ]:
!python -m eval.run_eval --config {CONFIGS} --split dev --repeat {DEV_REPEAT}

## 7. Test: `CONFIGS` × 1 lần
Chỉ chạy sau khi đã xem kết quả dev (tuần 4: chốt ngưỡng căn chỉnh B1 bằng `eval.metrics --align-sweep`; tuần 5: xem cột `qp` trong log dev, nhận diện sản phẩm và fallback). Test chạy một lần cho mỗi cấu hình.

Tuần 6: trên WSL chạy `PYTHONPATH=. python -m eval.metrics runs/B4_v1_dev_<stamp>_r* --gate-sweep`, chọn `rerank.min_score`, ghi vào `configs/B4.yaml`, commit + push, rồi chạy lại từ cell 3 (clone lại code) trước cell này. Cell dưới dừng nếu B4.yaml vẫn để `min_score: null`.

In [ ]:
import yaml
assert yaml.safe_load(open("configs/B4.yaml"))["rerank"]["min_score"] is not None, \
    "B4.yaml còn min_score: null: chọn ngưỡng trên dev (eval.metrics --gate-sweep) trước khi chạy test"
!python -m eval.run_eval --config {TEST_CONFIGS} --split test

## 8. Tóm tắt các run trên Drive

In [ ]:
import glob, json
for p in sorted(glob.glob(f"{DRIVE_DIR}/runs/*/meta.json")):
    m = json.load(open(p))
    print(f"{m['run_id']:40s} {m.get('gpu')}  LLM {m.get('llm_gpu_pct')}%  {m.get('n_logged')}/{m.get('n')} câu  "
          f"{m.get('wall_s')} s  ctx_overflow {m.get('ctx_overflow')}  commit {m.get('repo_commit')}")

## 9. Tắt runtime
Runtime vẫn tính unit khi không chạy gì. Log đã nằm trên Drive.

In [ ]:
from google.colab import runtime
if input("Tắt runtime? (y/n) ").strip().lower() == "y":
    runtime.unassign()